In [3]:
"""
Devoir : Extraction de texte depuis un PDF arabe et traitement NLP de base
=========================================================================
PDF utilise : document_arabe.pdf (Intelligence Artificielle et NLP en arabe)

PROBLEME RESOLU : Les PDF arabes exportes avec certains outils stockent
les caracteres sous deux formes defectueuses :
  1. "Presentation Forms" Unicode (FB50-FEFF) → corriges par NFKC
  2. Caracteres de chaque mot ecrits en ordre inverse → corriges par [::-1]
Ces deux corrections sont indispensables avant tout traitement NLP.
"""

import pdfplumber
import unicodedata
import re
from collections import Counter




def extract_text_from_pdf(pdf_path):
    """
    Extrait le texte de toutes les pages d'un PDF arabe.

    Deux corrections appliquees a l'extraction :
    - Normalisation NFKC : convertit les Presentation Forms arabes
      (plage FB50-FEFF) en caracteres arabes standard (0600-06FF)
    - Inversion de chaque mot : certains PDF stockent les caracteres
      de chaque mot dans l'ordre inverse (ex: 'ةيعيبطلا' -> 'الطبيعية')
    """
    texte_complet = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            texte_page = page.extract_text() or ""

            # Correction 1 : normalisation Unicode
            texte_page = unicodedata.normalize('NFKC', texte_page)

            # Correction 2 : inverser les caracteres de chaque mot
            mots_corriges = [mot[::-1] for mot in texte_page.split()]
            texte_page = ' '.join(mots_corriges)

            texte_complet += texte_page + "\n"
            print(f"[Page {i}] {len(texte_page)} caracteres extraits.")
    return texte_complet




def nettoyer_texte_arabe(texte):
    """
    Nettoie le texte arabe extrait :
    - Supprime les diacritiques (harakat / tashkeel)
    - Supprime les caracteres non-arabes (ponctuation, chiffres...)
    - Normalise les espaces
    """
    # Supprimer les diacritiques arabes
    diacritiques = re.compile(r'[\u064B-\u065F\u0670]')
    texte = diacritiques.sub('', texte)

    # Garder uniquement les lettres arabes et les espaces
    texte = re.sub(r'[^\u0600-\u06FF\s]', ' ', texte)

    # Normaliser les espaces multiples
    texte = re.sub(r'\s+', ' ', texte).strip()

    return texte



def tokeniser(texte):
    """Decoupe le texte en mots arabes (longueur >= 2 caracteres)."""
    return [mot for mot in texte.split() if len(mot) >= 2]


STOP_WORDS_AR = {
    'في', 'من', 'على', 'إلى', 'عن', 'مع', 'هذه', 'هذا', 'التي', 'الذي',
    'أن', 'إن', 'كما', 'لا', 'ما', 'أو', 'ثم', 'قد', 'كان', 'هو', 'هي',
    'لم', 'مما', 'بشكل', 'حيث', 'أي', 'يعد', 'ذلك', 'كبيرة', 'بسرعة',
    'وال', 'وأن', 'يكون', 'هذا', 'وهو', 'إذ', 'حين', 'بين', 'عند',
}

def supprimer_stop_words(mots, stop_words=STOP_WORDS_AR):
    """Supprime les mots vides de la liste de tokens."""
    return [mot for mot in mots if mot not in stop_words]




def analyser_frequence(mots, top_n=10):
    """Retourne les N mots les plus frequents avec leur compte."""
    return Counter(mots).most_common(top_n)



MOTS_CLES_THEMES = {
    'Intelligence Artificielle' : ['الذكاء', 'الاصطناعي'],
    'Apprentissage automatique' : ['التعلم', 'الآلي', 'الخوارزميات', 'العميق'],
    'NLP - Traitement du langage': ['اللغة', 'الطبيعية', 'معالجة', 'العربية'],
    'Reseaux de neurones'       : ['العصبية', 'الشبكات', 'نماذج'],
    'Applications sectorielles' : ['الطب', 'التعليم', 'الاقتصاد', 'تطبيقات'],
}

def detecter_themes(mots):
    """Detecte et affiche les themes presents dans le texte."""
    print("\nThemes detectes dans le document :")
    for theme, mots_cles in MOTS_CLES_THEMES.items():
        score = sum(mots.count(mc) for mc in mots_cles)
        barre = '#' * score if score else '-'
        statut = '[OK]' if score else '[  ]'
        print(f"  {statut} {theme:<35s} {barre} ({score})")



def pipeline_nlp_arabe(pdf_path):
    print("=" * 58)
    print("  PIPELINE NLP - DOCUMENT ARABE")
    print("=" * 58)

    # Etape 1 : Extraction + correction
    print("\n[Etape 1] Extraction et correction du texte PDF")
    texte_brut = extract_text_from_pdf(pdf_path)
    print(f"-> Total extrait : {len(texte_brut)} caracteres")
    print("\nApercu (40 premiers mots) :")
    print(' '.join(texte_brut.split()[:40]))

    # Etape 2 : Nettoyage
    print("\n\n[Etape 2] Nettoyage du texte arabe")
    texte_propre = nettoyer_texte_arabe(texte_brut)
    print(f"-> Apres nettoyage : {len(texte_propre)} caracteres")

    # Etape 3 : Tokenisation
    print("\n[Etape 3] Tokenisation")
    mots = tokeniser(texte_propre)
    print(f"-> Nombre de tokens : {len(mots)}")
    print(f"-> Exemples        : {mots[:8]}")

    # Etape 4 : Stop words
    print("\n[Etape 4] Suppression des mots vides")
    mots_filtres = supprimer_stop_words(mots)
    supprimes = len(mots) - len(mots_filtres)
    print(f"-> Avant : {len(mots)} | Apres : {len(mots_filtres)} ({supprimes} mots supprimes)")

    # Etape 5 : Frequences
    print("\n[Etape 5] Mots les plus frequents (Top 10)")
    print(f"  {'Rang':<5} {'Mot':<25} {'Freq':>5}  Visualisation")
    print("  " + "-" * 50)
    top_mots = analyser_frequence(mots_filtres, top_n=10)
    for rang, (mot, freq) in enumerate(top_mots, 1):
        barre = '|' * freq
        print(f"  {rang:<5} {mot:<25} {freq:>5}  {barre}")

    detecter_themes(mots_filtres)


    return {
        "texte_brut"   : texte_brut,
        "texte_propre" : texte_propre,
        "mots"         : mots,
        "mots_filtres" : mots_filtres,
        "top_mots"     : top_mots,
    }




if __name__ == "__main__":
    PDF_PATH = "document_arabe.pdf"   
    resultats = pipeline_nlp_arabe(PDF_PATH)

  PIPELINE NLP - DOCUMENT ARABE

[Etape 1] Extraction et correction du texte PDF
[Page 1] 954 caracteres extraits.
[Page 2] 601 caracteres extraits.
-> Total extrait : 1557 caracteres

Apercu (40 premiers mots) :
الطبيعية اللغة ومعالجة االصطناعي الذكاء مقدمة والعلمية. اليومية الحياة مجاالت مختلف في جذريا تحوال أحدث مما كبيرة بسرعة التقنية هذه تطورت والعشرين. الحادي القرن في العالم شهدها التي التقنيات أبرز من االصطناعي الذكاء يعد اكتشافها. اإلنسان على يصعب التي الخفية


[Etape 2] Nettoyage du texte arabe
-> Apres nettoyage : 1538 caracteres

[Etape 3] Tokenisation
-> Nombre de tokens : 233
-> Exemples        : ['الطبيعية', 'اللغة', 'ومعالجة', 'االصطناعي', 'الذكاء', 'مقدمة', 'والعلمية', 'اليومية']

[Etape 4] Suppression des mots vides
-> Avant : 233 | Apres : 191 (42 mots supprimes)

[Etape 5] Mots les plus frequents (Top 10)
  Rang  Mot                        Freq  Visualisation
  --------------------------------------------------
  1     اللغة                         6  ||||||
  2     